# Can we predict 12-month loan default?
**Question:** using only information available at application time, can we predict `defaulted` (1 = default within 12 months of funding)?

**Plan:** inspect the data → check for leakage/quality issues → hold out a test set → compare a base-rate baseline, logistic regression and gradient boosting using cross-validation → evaluate once on the test set → look at calibration, thresholds and drivers.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve, precision_recall_curve, brier_score_loss, confusion_matrix
from sklearn.calibration import calibration_curve
from sklearn.inspection import permutation_importance
SEED=0
df=pd.read_csv('data.csv'); dd=pd.read_csv('columns.csv')
dd

## 1. Data quality

In [ ]:
print(df.shape)
print('missing:', df.isna().sum().sum(), '| duplicate rows:', df.duplicated().sum(), '| duplicate ids:', df.application_id.duplicated().sum())
df.describe().T

In [ ]:
rate=df.defaulted.mean()
print(f'default rate: {rate:.3%} ({df.defaulted.sum()} of {len(df)})')

All columns are application-time attributes per the data dictionary, so there is no obvious target leakage. `application_id` is an identifier, not a feature, and is excluded. The data has no date column, so a time-based split isn't possible; a random stratified split is used, which assumes future applicants resemble these.

The classes are imbalanced (~10% defaults), so accuracy is misleading (always predicting "no default" gets ~90%). I use ROC AUC and average precision (PR AUC; the no-skill value equals the default rate).

## 2. Exploration

In [ ]:
num=['income_k','debt_ratio','credit_lines','age','employment_years','loan_amount_k']
fig,ax=plt.subplots(2,3,figsize=(13,7))
for a,col in zip(ax.ravel(),num):
    q=pd.qcut(df[col],10,duplicates='drop')
    g=df.groupby(q,observed=True).defaulted.mean()
    a.bar(range(len(g)),g.values,color='#4c78a8'); a.axhline(rate,color='k',ls='--',lw=1)
    a.set_title(col); a.set_xlabel('decile (low → high)'); a.set_ylabel('default rate')
plt.tight_layout(); plt.show()
df.groupby('region').defaulted.agg(['mean','size'])

## 3. Split and models

In [ ]:
X=df.drop(columns=['application_id','defaulted']); y=df.defaulted
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=0.25,stratify=y,random_state=SEED)
cat=['region']; num=[c for c in X if c not in cat]
logit=Pipeline([('pre',ColumnTransformer([('n',StandardScaler(),num),('c',OneHotEncoder(),cat)])),
                ('m',LogisticRegression(max_iter=1000))])
gbm=Pipeline([('pre',ColumnTransformer([('n','passthrough',num),('c',OneHotEncoder(),cat)])),
              ('m',HistGradientBoostingClassifier(max_depth=3,learning_rate=0.05,max_iter=200,l2_regularization=1.0,random_state=SEED))])
models={'baseline (base rate)':DummyClassifier(strategy='prior'),'logistic regression':logit,'gradient boosting':gbm}
cv=StratifiedKFold(5,shuffle=True,random_state=SEED)
rows=[]
for n,m in models.items():
    for metric in ['roc_auc','average_precision']:
        s=cross_val_score(m,Xtr,ytr,cv=cv,scoring=metric)
        rows.append((n,metric,s.mean(),s.std()))
pd.DataFrame(rows,columns=['model','metric','cv mean','cv std']).round(3)

## 4. Held-out test evaluation (used once)

In [ ]:
fitted={n:m.fit(Xtr,ytr) for n,m in models.items()}
proba={n:m.predict_proba(Xte)[:,1] for n,m in fitted.items()}
def boot_auc_ci(p,n=500):
    rng=np.random.default_rng(SEED); yy=yte.values; a=[]
    for _ in range(n):
        i=rng.integers(0,len(yy),len(yy)); a.append(roc_auc_score(yy[i],p[i]))
    return np.percentile(a,[2.5,97.5])
res=[]
for n,p in proba.items():
    ci='n/a' if n.startswith('baseline') else '[%.3f, %.3f]'%tuple(boot_auc_ci(p))
    res.append((n,roc_auc_score(yte,p),ci,average_precision_score(yte,p),brier_score_loss(yte,p)))
pd.DataFrame(res,columns=['model','ROC AUC','AUC 95% CI (bootstrap)','PR AUC','Brier']).round(3).set_index('model')

In [ ]:
fig,ax=plt.subplots(1,2,figsize=(12,4.5))
for n,p in proba.items():
    f,t,_=roc_curve(yte,p); ax[0].plot(f,t,label=n)
    pr,rc,_=precision_recall_curve(yte,p); ax[1].plot(rc,pr,label=n)
ax[0].plot([0,1],[0,1],'k:',lw=1); ax[0].set(xlabel='false positive rate',ylabel='true positive rate',title='ROC')
ax[1].axhline(yte.mean(),color='k',ls=':',lw=1); ax[1].set(xlabel='recall',ylabel='precision',title='Precision–recall')
ax[0].legend(); plt.tight_layout(); plt.show()

## 5. Calibration and ranking value

In [ ]:
best='logistic regression' if roc_auc_score(yte,proba['logistic regression'])>=roc_auc_score(yte,proba['gradient boosting']) else 'gradient boosting'
print('best on test:',best)
p=proba[best]
fp,mp=calibration_curve(yte,p,n_bins=8,strategy='quantile')
plt.figure(figsize=(4.5,4.5)); plt.plot(mp,fp,'o-'); plt.plot([0,.4],[0,.4],'k:'); plt.xlabel('predicted probability'); plt.ylabel('observed default rate'); plt.title('Calibration'); plt.show()
d=pd.DataFrame({'p':p,'y':yte.values}); d['decile']=pd.qcut(d.p.rank(method='first'),10,labels=range(1,11))
lift=d.groupby('decile',observed=True).y.agg(['mean','sum']).rename(columns={'mean':'default rate','sum':'defaults'})
lift['lift vs base']=lift['default rate']/yte.mean(); lift.round(3)

## 6. Choosing a threshold

In [ ]:
rows=[]
for t in [0.05,0.1,0.15,0.2,0.3]:
    pred=p>=t; tn,fp_,fn,tp=confusion_matrix(yte,pred).ravel()
    rows.append((t,pred.mean(),tp/(tp+fn),tp/max(tp+fp_,1)))
pd.DataFrame(rows,columns=['threshold','share flagged','recall','precision']).round(3)

The right cut-off depends on the relative cost of a missed default vs. a rejected good applicant, which isn't in the data. The table shows the trade-off.

## 7. What drives predictions?

In [ ]:
pi=permutation_importance(fitted[best],Xte,yte,scoring='roc_auc',n_repeats=20,random_state=SEED)
imp=pd.Series(pi.importances_mean,index=X.columns).sort_values(); err=pd.Series(pi.importances_std,index=X.columns)[imp.index]
imp.plot.barh(xerr=err,figsize=(6,3.5)); plt.xlabel('drop in test ROC AUC when shuffled'); plt.show()

In [ ]:
lr=fitted['logistic regression']; names=lr.named_steps['pre'].get_feature_names_out()
pd.Series(lr.named_steps['m'].coef_[0],index=names).sort_values().round(3)

*Coefficients are per standard deviation for numeric features, so they are comparable in size.*

## Conclusions
(Numbers are from the run on this data with `SEED=0`; re-running the notebook regenerates them.)
- **Is there signal?** Yes. Test ROC AUC is about 0.78 (bootstrap 95% CI roughly 0.75–0.81) against 0.50 for a base-rate model, and 5-fold CV on the training set agrees (about 0.77). PR AUC is about 0.31 vs. a no-skill 0.10.
- **How strong is it?** Moderate. It ranks risk well: in the test set the riskiest decile of predictions defaulted about 35% of the time vs. about 0.5% in the safest decile (base rate 10.5%). Individual predictions are uncertain: flagging the top ~15% of applicants catches only ~45% of defaults with ~32% precision.
- **Model choice:** logistic regression (CV AUC 0.773) slightly beat gradient boosting (0.762), so the signal is largely additive and linear. The simpler, more interpretable model is the recommendation.
- **Drivers:** higher `debt_ratio` and `loan_amount_k` raise risk; higher `income_k` and `employment_years` lower it. `age` and `credit_lines` contribute essentially nothing, and `region` very little.

**Caveats:** random (not time-based) split because there is no date column; only funded loans are present, so there is selection bias and nothing is known about rejected applicants; the threshold needs business cost inputs; and `age` and `region` are in the data, so a fairness/regulatory review is needed before any use in credit decisions.